# Comfinator

In [ ]:
%%time
update = False
import os
import stat

home_dir = '/kaggle/working'
python = '/kaggle/working/venv/bin/python'
pip = '/kaggle/working/venv/bin/pip'

def find_bin_folders(folder_path):
    bin_folders = []
    for root, dirs, files in os.walk(folder_path):
        for dir_name in dirs:
            if dir_name == 'bin':
                bin_folders.append(os.path.join(root, dir_name))
    return bin_folders

def installLibraries(home_dir, python, pip):
    %cd {home_dir}
    
    print("=" * 70)
    print("INSTALLING PYTORCH 2.6.0 WITH CUDA 12.4")
    print("=" * 70)
    
    # Install PyTorch 2.6.0 with CUDA 12.4 (fixes CVE-2025-32434)
    !{pip} install torch==2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    
    # Verify PyTorch installation
    print("\n✅ Verifying PyTorch installation...")
    !{python} -c "import torch; print(f'PyTorch version: {torch.__version__}'); print(f'CUDA available: {torch.cuda.is_available()}'); print(f'CUDA version: {torch.version.cuda if torch.cuda.is_available() else \"N/A\"}')"
    
    !{pip} install tensorflow[and-cuda]
    
    # Install safetensors to bypass torch.load security issues
    !{pip} install safetensors
    !wget https://q4j3.c11.e2-5.dev/downloads/req.txt
    !{pip} install -r /kaggle/working/req.txt
    
    print("\n" + "=" * 70)
    print("✅ PyTorch 2.6.0+cu124 installed successfully!")
    print("🔒 CVE-2025-32434 vulnerability FIXED!")
    print("=" * 70)

!pip install virtualenv

if not os.path.exists(f'{home_dir}/venv'):
    print('Installing venv')
    os.chdir(home_dir)
    get_ipython().system(f'cd {home_dir}')
    get_ipython().system('virtualenv venv -p $(which python3.10)')
    installLibraries(home_dir, python, pip)
else:
    bin_folders = find_bin_folders('/kaggle/working/venv')
    if bin_folders:
        print("Found 'bin' folders:")
        for bin_folder in bin_folders:
            print(bin_folder)
            for filename in os.listdir(bin_folder):
                file_path = os.path.join(bin_folder, filename)
                if os.path.isfile(file_path):
                    current_permissions = os.stat(file_path).st_mode
                    os.chmod(file_path, current_permissions | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

if not os.path.exists(f'{home_dir}/venv/bin/python3.10'):
    get_ipython().system('cp /usr/bin/python3.10 /kaggle/working/venv/bin/')

!ln -s /kaggle/working/venv/bin/python3.10 /kaggle/working/venv/bin/python
!ln -s /kaggle/working/venv/bin/python3.10 /kaggle/working/venv/bin/python3

%cd /kaggle/working
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!git checkout v0.3.62  # Use latest stable release tag
!{pip} install -r requirements.txt

# Redirect all HuggingFace downloads to /tmp
os.environ['HF_HOME'] = '/tmp/huggingface'
os.environ['HF_HUB_CACHE'] = '/tmp/huggingface/hub'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/huggingface/transformers'
os.environ['DIFFUSERS_CACHE'] = '/tmp/huggingface/diffusers'

# Redirect torch hub cache to /tmp
os.environ['TORCH_HOME'] = '/tmp/torch'

# Redirect XDG cache (used by many Python libraries) to /tmp
os.environ['XDG_CACHE_HOME'] = '/tmp/cache'

# Create the directories
!mkdir -p /tmp/huggingface/hub
!mkdir -p /tmp/huggingface/diffusers
!mkdir -p /tmp/torch
!mkdir -p /tmp/cache

# Create temp directories for ALL model types (including ControlNet and Diffusers)
!mkdir -p /tmp/models/checkpoints
!mkdir -p /tmp/models/clip
!mkdir -p /tmp/models/vae
!mkdir -p /tmp/models/unet
!mkdir -p /tmp/models/diffusion_models
!mkdir -p /tmp/models/controlnet
!mkdir -p /tmp/models/loras
!mkdir -p /tmp/models/upscale_models
!mkdir -p /tmp/models/diffusers

# Remove existing model directories
!rm -rf /kaggle/working/ComfyUI/models/checkpoints
!rm -rf /kaggle/working/ComfyUI/models/diffusion_models
!rm -rf /kaggle/working/ComfyUI/models/clip
!rm -rf /kaggle/working/ComfyUI/models/vae
!rm -rf /kaggle/working/ComfyUI/models/unet
!rm -rf /kaggle/working/ComfyUI/models/controlnet
!rm -rf /kaggle/working/ComfyUI/models/loras
!rm -rf /kaggle/working/ComfyUI/models/upscale_models
!rm -rf /kaggle/working/ComfyUI/models/diffusers

# Link all model directories to temp storage
!ln -s /tmp/models/checkpoints /kaggle/working/ComfyUI/models/checkpoints
!ln -s /tmp/models/diffusion_models /kaggle/working/ComfyUI/models/diffusion_models
!ln -s /tmp/models/clip /kaggle/working/ComfyUI/models/clip
!ln -s /tmp/models/vae /kaggle/working/ComfyUI/models/vae
!ln -s /tmp/models/unet /kaggle/working/ComfyUI/models/unet
!ln -s /tmp/models/controlnet /kaggle/working/ComfyUI/models/controlnet
!ln -s /tmp/models/loras /kaggle/working/ComfyUI/models/loras
!ln -s /tmp/models/upscale_models /kaggle/working/ComfyUI/models/upscale_models
!ln -s /tmp/models/diffusers /kaggle/working/ComfyUI/models/diffusers

# Additional temp models directory
checkpoints = '/kaggle/working/ComfyUI/models/checkpoints'
link_path = checkpoints + '/temp-models'
temp_models = '/kaggle/temp/temp-models'

!mkdir -p /kaggle/temp
!mkdir -p $temp_models

if not os.path.exists(link_path):
    get_ipython().system(f'ln -s {temp_models} {checkpoints}')

# Install ComfyUI Manager
update_manager = True
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ComfyUI-Manager
!git pull

if update_manager:
    get_ipython().system('git pull')

# Download utilities
!wget https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O /kaggle/working/pinggy.py

# GPU offload utility
%cd /kaggle/working/ComfyUI/custom_nodes
!wget https://gist.githubusercontent.com/city96/30743dfdfe129b331b5676a79c3a8a39/raw/ecb4f6f5202c20ea723186c93da308212ba04cfb/ComfyBootlegOffload.py

print("\n" + "=" * 70)
print("✅ SETUP COMPLETE!")
print("=" * 70)

## Zrok

In [ ]:
import os

#============================
ZROK_TOKEN = "jp9EY3XaOGfm"  # Replace with your zrok token
#============================

PORT_TO_SHARE = 8188      

!mkdir -p /kaggle/working/zrok
%cd /kaggle/working/zrok

if not os.path.exists('/kaggle/working/zrok/zrok'):
    print("Downloading and setting up zrok...")
    !wget https://github.com/openziti/zrok/releases/download/v1.0.4/zrok_1.0.4_linux_amd64.tar.gz -O zrok.tar.gz
    !tar -xvf ./zrok.tar.gz
    !chmod a+x /kaggle/working/zrok/zrok
else:
    print("zrok is already set up.")

!/kaggle/working/zrok/zrok enable {ZROK_TOKEN}

In [ ]:
# Fix matplotlib backend issue first
import os
os.environ['MPLBACKEND'] = 'Agg'
print("DONE.")

# Download upscale model

In [ ]:
import os
import requests

# URL of the model file
url = "https://huggingface.co/FacehugmanIII/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.pth"

# New path to save the model
folder_path = "/kaggle/working/ComfyUI/models/upscale_models"
file_name = "4x_foolhardy_Remacri.pth"
file_path = os.path.join(folder_path, file_name)

# Create directory if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

# Download the file
response = requests.get(url, stream=True)
response.raise_for_status()

with open(file_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print(f"Downloaded model saved to: {file_path}")


 # 3D "stuff" setup

In [ ]:
#!/usr/bin/env python3
"""
Complete Hunyuan3D-2 Setup for Kaggle
- Downloads model from Kijai's repo
- Installs ComfyUI-Hunyuan3DWrapper
- Installs ComfyUI_essentials + transparent-background
- Fixes HuggingFace cache/storage issues
- Compiles custom_rasterizer CUDA extension
"""

import os
import sys
import subprocess
import shutil

print("=" * 70)
print("HUNYUAN3D-2 COMPLETE SETUP FOR KAGGLE")
print("=" * 70)

# ============================================================================
# STEP 1: FIX STORAGE - Redirect HuggingFace cache to /tmp
# ============================================================================
print("\n[1/8] Fixing storage configuration...")

# Clear existing cache to free space
cache_dirs = ['/root/.cache/huggingface', '/root/.cache/torch']
total_freed = 0

for cache_dir in cache_dirs:
    if os.path.exists(cache_dir):
        try:
            size = sum(
                os.path.getsize(os.path.join(dp, f))
                for dp, dn, filenames in os.walk(cache_dir)
                for f in filenames
            ) / (1024**3)
            shutil.rmtree(cache_dir)
            total_freed += size
            print(f"✓ Cleared {cache_dir}: {size:.2f} GB freed")
        except Exception as e:
            print(f"⚠ Could not clear {cache_dir}: {e}")

# Redirect all HuggingFace operations to /tmp (70GB+ available)
os.environ['HF_HOME'] = '/tmp/huggingface'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/huggingface/transformers'
os.environ['HF_HUB_CACHE'] = '/tmp/huggingface/hub'
os.environ['TORCH_HOME'] = '/tmp/torch'
os.makedirs('/tmp/huggingface', exist_ok=True)
os.makedirs('/tmp/torch', exist_ok=True)

# Check available space
total, used, free = shutil.disk_usage('/kaggle/working')
total_tmp, used_tmp, free_tmp = shutil.disk_usage('/tmp')

print(f"\n📊 Storage Status:")
print(f"   /kaggle/working: {free / (1024**3):.2f} GB free (saved permanently)")
print(f"   /tmp: {free_tmp / (1024**3):.2f} GB free (temporary)")
print(f"   Cache freed: {total_freed:.2f} GB")
print(f"✓ HuggingFace cache redirected to /tmp")

# ============================================================================
# STEP 2: Setup ComfyUI directory structure
# ============================================================================
print("\n[2/8] Setting up ComfyUI directories...")

comfy_base = '/kaggle/working/ComfyUI'
models_dir = os.path.join(comfy_base, 'models', 'diffusion_models')
checkpoints_dir = os.path.join(comfy_base, 'models', 'checkpoints')
custom_nodes = os.path.join(comfy_base, 'custom_nodes')

for directory in [models_dir, checkpoints_dir, custom_nodes]:
    os.makedirs(directory, exist_ok=True)
    print(f"✓ {directory}")

# ============================================================================
# STEP 3: Download Hunyuan3D-2 Model
# ============================================================================
print("\n[3/8] Downloading Hunyuan3D-2 Model...")

model_url = 'https://huggingface.co/Kijai/Hunyuan3D-2_safetensors/resolve/main/hunyuan3d-dit-v2-0-fp16.safetensors'
model_path = os.path.join(models_dir, 'hunyuan3d-dit-v2-0-fp16.safetensors')

# Check if already downloaded
if os.path.exists(model_path):
    existing_size = os.path.getsize(model_path) / (1024**3)
    if existing_size > 4.5:  # Should be ~4.93GB
        print(f"✓ Model already exists ({existing_size:.2f} GB)")
        print(f"  Location: {model_path}")
    else:
        print(f"⚠ Removing incomplete file ({existing_size:.2f} GB)")
        os.remove(model_path)

if not os.path.exists(model_path) or os.path.getsize(model_path) < 4.5 * 1024**3:
    print(f"Downloading model (4.93 GB)...")
    print("This will take 5-10 minutes on Kaggle GPU...")
    
    try:
        # Use wget with optimal flags
        result = subprocess.run([
            'wget',
            '--no-check-certificate',
            '--continue',
            '--progress=dot:giga',
            '--timeout=900',
            '--tries=3',
            '--retry-connrefused',
            '-O', model_path,
            model_url
        ], timeout=1800)
        
        if result.returncode == 0 and os.path.exists(model_path):
            file_size = os.path.getsize(model_path) / (1024**3)
            
            if file_size > 4.5:
                print(f"\n✓ Model downloaded successfully: {file_size:.2f} GB")
            else:
                print(f"\n✗ Download incomplete: {file_size:.2f} GB")
                raise Exception("Incomplete download")
        else:
            raise Exception(f"wget failed with code {result.returncode}")
            
    except Exception as e:
        print(f"\n✗ Download failed: {e}")
        print("\n⚠ MANUAL DOWNLOAD REQUIRED:")
        print(f"   1. Visit: https://huggingface.co/Kijai/Hunyuan3D-2_safetensors")
        print(f"   2. Download: hunyuan3d-dit-v2-0-fp16.safetensors")
        print(f"   3. Upload to Kaggle and place in: {models_dir}")
        sys.exit(1)

# ============================================================================
# STEP 4: Clone ComfyUI-Hunyuan3DWrapper
# ============================================================================
print("\n[4/8] Installing ComfyUI-Hunyuan3DWrapper...")

wrapper_dir = os.path.join(custom_nodes, 'ComfyUI-Hunyuan3DWrapper')

if os.path.exists(wrapper_dir):
    print(f"✓ Wrapper already exists, updating...")
    try:
        subprocess.run(['git', '-C', wrapper_dir, 'pull'], check=True, capture_output=True)
        print("✓ Updated to latest version")
    except:
        print("⚠ Could not update, using existing version")
else:
    print(f"Cloning wrapper repository...")
    try:
        subprocess.run([
            'git', 'clone',
            'https://github.com/kijai/ComfyUI-Hunyuan3DWrapper.git',
            wrapper_dir
        ], check=True)
        print(f"✓ Cloned successfully")
    except Exception as e:
        print(f"✗ Clone failed: {e}")
        sys.exit(1)

# ============================================================================
# STEP 5: Install ComfyUI_essentials
# ============================================================================
print("\n[5/8] Installing ComfyUI_essentials...")

essentials_dir = os.path.join(custom_nodes, 'ComfyUI_essentials')

if os.path.exists(essentials_dir):
    print(f"✓ ComfyUI_essentials already exists, updating...")
    try:
        subprocess.run(['git', '-C', essentials_dir, 'pull'], check=True, capture_output=True)
        print("✓ Updated to latest version")
    except:
        print("⚠ Could not update, using existing version")
else:
    print(f"Cloning ComfyUI_essentials repository...")
    try:
        subprocess.run([
            'git', 'clone',
            'https://github.com/cubiq/ComfyUI_essentials.git',
            essentials_dir
        ], check=True)
        print(f"✓ Cloned successfully")
    except Exception as e:
        print(f"✗ Clone failed: {e}")
        print("⚠ Continuing without ComfyUI_essentials...")

# ============================================================================
# STEP 6: Install Background Removal Dependencies
# ============================================================================
print("\n[6/8] Installing background removal packages...")

# Fix NumPy/SciPy compatibility issues - CRITICAL FIX
print("Fixing NumPy/SciPy/Albumentations compatibility...")
try:
    # First, uninstall broken scipy
    subprocess.run([
        sys.executable, '-m', 'pip', 'uninstall', 'scipy', '-y'
    ], capture_output=True)
    
    # Install compatible versions together
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==1.26.4',  # Stable NumPy 1.x version
        'scipy==1.13.1',  # Compatible SciPy version
        '--force-reinstall',
        '--no-cache-dir',
        '--no-warn-script-location'
    ], timeout=300)
    print("✓ NumPy 1.26.4 and SciPy 1.13.1 installed")
    
    # Now install albumentations with compatible opencv
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'opencv-python-headless>=4.8.0',
        'albumentations>=1.3.0',
        '--no-warn-script-location',
        '--quiet'
    ], timeout=300)
    print("✓ Albumentations and dependencies installed")
    
except Exception as e:
    print(f"⚠ Warning during dependency fix: {str(e)[:200]}")
    print("  Continuing anyway...")

bg_packages = [
    'transparent-background',  # For TransparentBGSession+
    'rembg',                   # Alternative background removal
]

for package in bg_packages:
    try:
        print(f"Installing {package}...")
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install',
            package,
            '--no-deps',  # Skip dependencies to avoid conflicts
            '--no-warn-script-location',
            '--quiet'
        ])
        
        # Install missing deps manually
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install',
            package,
            '--no-warn-script-location',
            '--quiet'
        ])
        print(f"✓ {package} installed")
    except Exception as e:
        print(f"⚠ Failed to install {package}: {str(e)[:100]}")
        print(f"  (Will try to install on first use)")

# Verify transparent-background installation
print("\nVerifying installation...")
try:
    import numpy
    import scipy
    print(f"✓ NumPy {numpy.__version__}")
    print(f"✓ SciPy {scipy.__version__}")
    
    from transparent_background import Remover
    print("✓ transparent-background verified and ready")
except ImportError as e:
    print(f"⚠ Import test failed: {str(e)[:150]}")
    print("  Package installed but needs kernel restart to work properly")
except Exception as e:
    print(f"⚠ Unexpected error: {str(e)[:150]}")
    print("  Package installed but needs kernel restart to work properly")

# ============================================================================
# STEP 7: Install Main Dependencies
# ============================================================================
print("\n[7/8] Installing Python dependencies...")

# First, upgrade transformers and diffusers to latest versions for compatibility
print("Upgrading transformers and diffusers to latest versions...")
try:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'transformers>=4.46.0',  # Supports offload_state_dict parameter
        'diffusers>=0.31.0',     # Latest stable diffusers
        'accelerate>=0.34.0',    # Required for model offloading
        '--upgrade',
        '--no-warn-script-location',
        '--quiet'
    ], timeout=300)
    print("✓ Upgraded transformers, diffusers, and accelerate")
except Exception as e:
    print(f"⚠ Upgrade warning: {str(e)[:150]}")
    print("  Continuing anyway...")

# Install Hunyuan3DWrapper requirements
requirements_file = os.path.join(wrapper_dir, 'requirements.txt')

if os.path.exists(requirements_file):
    try:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install',
            '-r', requirements_file,
            '--no-warn-script-location',
            '--quiet'
        ])
        print("✓ Hunyuan3DWrapper dependencies installed")
    except Exception as e:
        print(f"⚠ Some dependencies may have failed: {e}")
        print("  This is usually OK - missing dependencies will auto-install on first use")
else:
    print(f"⚠ requirements.txt not found at {requirements_file}")

# Install ComfyUI_essentials requirements if available
essentials_req = os.path.join(essentials_dir, 'requirements.txt')
if os.path.exists(essentials_req):
    try:
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install',
            '-r', essentials_req,
            '--no-warn-script-location',
            '--quiet'
        ])
        print("✓ ComfyUI_essentials dependencies installed")
    except Exception as e:
        print(f"⚠ ComfyUI_essentials requirements installation issue: {e}")

# ============================================================================
# STEP 7.5: Patch Hunyuan3D nodes.py for compatibility
# ============================================================================
print("\n[7.5/8] Patching Hunyuan3D nodes for compatibility...")

nodes_file = os.path.join(wrapper_dir, 'nodes.py')
if os.path.exists(nodes_file):
    try:
        # Read the file
        with open(nodes_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Check if patch is needed
        if 'offload_state_dict=True' in content or 'offload_state_dict = True' in content:
            print("Removing offload_state_dict parameter...")
            
            # Remove offload_state_dict parameter from from_pretrained calls
            import re
            content = re.sub(r',?\s*offload_state_dict\s*=\s*True', '', content)
            
            # Write back
            with open(nodes_file, 'w', encoding='utf-8') as f:
                f.write(content)
            
            print("✓ nodes.py patched successfully")
        else:
            print("✓ nodes.py doesn't need patching")
            
    except Exception as e:
        print(f"⚠ Could not patch nodes.py: {e}")
        print("  You may encounter compatibility issues")
else:
    print(f"⚠ nodes.py not found at {nodes_file}")

# ============================================================================
# STEP 8: Compile custom_rasterizer CUDA Extension
# ============================================================================
print("\n[8/8] Compiling custom_rasterizer CUDA extension...")

rasterizer_dir = os.path.join(wrapper_dir, 'hy3dgen', 'texgen', 'custom_rasterizer')

if os.path.exists(rasterizer_dir):
    print(f"✓ Found custom_rasterizer directory: {rasterizer_dir}")
    
    # Check if already compiled
    try:
        sys.path.insert(0, os.path.dirname(rasterizer_dir))
        import custom_rasterizer
        print("✓ custom_rasterizer already compiled and working!")
    except ImportError:
        print("⚙️  Compiling custom_rasterizer (this may take 2-3 minutes)...")
        
        # Install ninja for faster compilation
        try:
            subprocess.check_call([
                sys.executable, '-m', 'pip', 'install', 'ninja',
                '--no-warn-script-location', '--quiet'
            ])
            print("✓ Ninja build system installed")
        except:
            print("⚠ Could not install ninja, using default compiler")
        
        # Compile the extension
        setup_py = os.path.join(rasterizer_dir, 'setup.py')
        
        if os.path.exists(setup_py):
            try:
                # Change to rasterizer directory
                original_dir = os.getcwd()
                os.chdir(rasterizer_dir)
                
                # Run setup.py build
                result = subprocess.run([
                    sys.executable, 'setup.py', 'build_ext', '--inplace'
                ], capture_output=True, text=True, timeout=300)
                
                os.chdir(original_dir)
                
                if result.returncode == 0:
                    print("✓ custom_rasterizer compiled successfully!")
                    
                    # Verify compilation
                    try:
                        sys.path.insert(0, os.path.dirname(rasterizer_dir))
                        import custom_rasterizer
                        print("✓ custom_rasterizer import verified!")
                    except ImportError as e:
                        print(f"⚠ Compilation succeeded but import failed: {e}")
                        print("  This may resolve on ComfyUI restart")
                else:
                    print(f"⚠ Compilation had issues:")
                    print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
                    print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
                    
            except subprocess.TimeoutExpired:
                print("⚠ Compilation timed out (this can happen on first build)")
                print("  The extension may compile on first use")
            except Exception as e:
                print(f"⚠ Compilation error: {e}")
                print("  The extension may auto-compile on first use")
        else:
            print(f"⚠ setup.py not found at {setup_py}")
            print("  The extension may auto-compile on first use")
else:
    print(f"⚠ custom_rasterizer directory not found at {rasterizer_dir}")
    print("  This may be due to repository structure changes")

# ============================================================================
# FINAL STATUS
# ============================================================================
print("\n" + "=" * 70)
print("SETUP COMPLETE!")
print("=" * 70)

# Verify installations
print("\n📦 Installed Components:")
print(f"   ✓ Model: {os.path.basename(model_path)} ({os.path.getsize(model_path) / (1024**3):.2f} GB)")
print(f"   ✓ Wrapper: ComfyUI-Hunyuan3DWrapper")
print(f"   ✓ Essentials: ComfyUI_essentials")
print(f"   ✓ Background removal: transparent-background, rembg")
print(f"   ✓ HuggingFace cache: redirected to /tmp")
print(f"   ✓ custom_rasterizer: compilation attempted")

print("\n📍 File Locations:")
print(f"   Model: {model_path}")
print(f"   Hunyuan3D Wrapper: {wrapper_dir}")
print(f"   ComfyUI Essentials: {essentials_dir}")
print(f"   Cache: /tmp/huggingface/")

# Storage check
total, used, free = shutil.disk_usage('/kaggle/working')
print(f"\n💾 Final Storage:")
print(f"   /kaggle/working free: {free / (1024**3):.2f} GB")

if free / (1024**3) > 10:
    print("   ✅ Sufficient space for Hunyuan3D operations")
else:
    print(f"   ⚠️ Low space! Recommended to free {10 - free / (1024**3):.2f} GB more")
print("\n" + "=" * 70)

# Build and install custom_rasterizer wheel for CUDA 12.4
print("\n[9/9] Building custom_rasterizer wheel for CUDA 12.4...")
rasterizer_dir = '/kaggle/working/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper/hy3dgen/texgen/custom_rasterizer'

import subprocess
original_dir = os.getcwd()
os.chdir(rasterizer_dir)

# Build wheel
result = subprocess.run(
    [python, 'setup.py', 'bdist_wheel'],
    capture_output=True,
    text=True,
    timeout=300
)

if result.returncode == 0:
    # Find and install the wheel
    import glob
    wheels = glob.glob(f'{rasterizer_dir}/dist/*.whl')
    if wheels:
        wheel_path = wheels[0]
        print(f"✓ Wheel built: {os.path.basename(wheel_path)}")
        
        # Install it
        install_result = subprocess.run(
            [python, '-m', 'pip', 'install', '--force-reinstall', wheel_path],
            capture_output=True,
            text=True
        )
        
        if install_result.returncode == 0:
            print("✓ custom_rasterizer wheel installed successfully!")
        else:
            print(f"⚠️ Wheel installation failed: {install_result.stderr[:200]}")
    else:
        print("⚠️ Wheel build succeeded but no .whl file found")
else:
    print(f"⚠️ Wheel build failed: {result.stderr[:200]}")

os.chdir(original_dir)


# Custom nodes

In [ ]:
import os
import time

# Setup directory
custom_nodes_dir = '/kaggle/working/ComfyUI/custom_nodes'
if not os.path.exists(custom_nodes_dir):
    custom_nodes_dir = './custom_nodes'
os.makedirs(custom_nodes_dir, exist_ok=True)

print("🚀 Installing ComfyUI Custom Nodes...")
%cd $custom_nodes_dir

repos = [
    "https://github.com/kijai/ComfyUI-Hunyuan3DWrapper.git",
    "https://github.com/crystian/ComfyUI-Crystools.git",
    "https://github.com/rgthree/rgthree-comfy.git",
    "https://github.com/yolain/ComfyUI-Easy-Use.git",
    "https://github.com/pollockjj/ComfyUI-MultiGPU",
]

for i, repo_url in enumerate(repos, 1):
    repo_name = repo_url.split('/')[-1].replace('.git','')
    print(f"[{i}/{len(repos)}] {repo_name}")
    
    if not os.path.exists(repo_name):
        try:
            get_ipython().system(f'git clone {repo_url} -q')
            print("  ✅ Installed")
        except Exception:
            print("  ❌ Failed")
    else:
        print("  ✅ Exists")
    
    time.sleep(1)

print("\n🎉 Nodes installation complete!")

## Fix Custom Node Dependencies
**Unified dependency installation for ALL custom nodes**

In [ ]:
import os
import subprocess
import sys
import importlib.util

print("🔧 Installing Dependencies...")

# Determine Python executable
venv_python = '/kaggle/working/venv/bin/python'
system_python = '/usr/bin/python3'

if os.path.exists(venv_python):
    try:
        result = subprocess.run([venv_python, '--version'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            PYTHON_EXEC = venv_python
            PIP_EXEC = '/kaggle/working/venv/bin/pip'
            print("✅ Using virtual environment")
        else:
            PYTHON_EXEC = system_python
            PIP_EXEC = 'pip3'
            print("⚠️ Using system Python")
    except:
        PYTHON_EXEC = system_python
        PIP_EXEC = 'pip3'
        print("⚠️ Using system Python")
else:
    PYTHON_EXEC = system_python
    PIP_EXEC = 'pip3'
    print("📝 Using system Python")

# Install custom node requirements
custom_nodes_dir = '/kaggle/working/ComfyUI/custom_nodes'
if os.path.exists(custom_nodes_dir):
    print("🔧 Installing custom node requirements...")
    for node_dir in os.listdir(custom_nodes_dir):
        node_path = os.path.join(custom_nodes_dir, node_dir)
        if os.path.isdir(node_path):
            requirements_file = os.path.join(node_path, 'requirements.txt')
            if os.path.exists(requirements_file):
                try:
                    subprocess.run([PIP_EXEC, 'install', '-r', requirements_file],
                                 capture_output=True, text=True, timeout=300)
                except:
                    pass

print("Done!")

In [ ]:
# Fix CLIPTextModel and tokenizers compatibility issues
pip = '/kaggle/working/venv/bin/pip'
python = '/kaggle/working/venv/bin/python'
!{pip} uninstall -y transformers tokenizers
!{pip} install transformers==4.44.2 tokenizers==0.19.1
!{pip} install diffusers==0.30.3 --upgrade
!{python} -c "import transformers; print(f'Transformers: {transformers.__version__}')"
!{python} -c "import tokenizers; print(f'Tokenizers: {tokenizers.__version__}')"
!{python} -c "import diffusers; print(f'Diffusers: {diffusers.__version__}')"
print("\n✅ All dependencies fixed! Restart kernel and re-run workflow.")

In [ ]:
import os
import subprocess
import time
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm
import requests

print("🚀 Model Downloader")
print("📥 Downloading models...")

# ============================================================================
# 🔑 HUGGINGFACE TOKEN (Optional)
# ============================================================================
# Only required for GATED models (e.g., FLUX.1-dev, SDXL-Turbo)
# 
# To add HF_TOKEN to Kaggle Secrets:
# 1. Get token from: https://huggingface.co/settings/tokens
# 2. In Kaggle: Add-ons → Secrets → Add a new secret
# 3. Label: HF_TOKEN
# 4. Value: Paste your token
# 
# If you're only downloading public models (SD1.5, SDXL, ControlNet),
# you don't need this token.
# ============================================================================

user_secrets = UserSecretsClient()
hf_token = None
try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("✅ HuggingFace token loaded (for gated models)")
except Exception:
    print("ℹ️  No HF_TOKEN found (only needed for gated models)")
    hf_token = os.getenv('HF_TOKEN')

base_dir = '/kaggle/working/ComfyUI'

def download_file_with_progress(url, destination):
    """Download file with progress bar using tqdm"""
    try:
        # Add HF token to headers if available and it's a HuggingFace URL
        headers = {}
        if hf_token and 'huggingface.co' in url:
            headers['Authorization'] = f'Bearer {hf_token}'
        
        response = requests.get(url, stream=True, timeout=300, headers=headers)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))
        
        os.makedirs(os.path.dirname(destination), exist_ok=True)
        
        with open(destination, 'wb') as file, tqdm(
            desc=os.path.basename(destination),
            total=total_size,
            unit='iB',
            unit_scale=True,
            unit_divisor=1024,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]'
        ) as progress_bar:
            for data in response.iter_content(chunk_size=8192):
                size = file.write(data)
                progress_bar.update(size)
        
        return True
    except Exception as e:
        print(f"Error: {e}")
        return False

# ============================================================================
# 📝 HOW TO ADD YOUR OWN DOWNLOADS:
# ============================================================================
# 1. Copy one of the example lines below
# 2. Change the 'name' to describe your model
# 3. Paste the download URL from CivitAI or HuggingFace
# 4. Give it a filename (must end with .safetensors or .pth or .gguf)
# 5. Choose the correct folder:
#    - 'checkpoints' = Main models (SD1.5, SDXL, FLUX, etc.)
#    - 'loras' = LoRA files
#    - 'controlnet' = ControlNet models
#    - 'upscale_models' = Upscalers (ESRGAN, etc.)
# 6. Remove the # at the start of the line to enable it
# ============================================================================

downloads = [
    # ==================== CHECKPOINTS (Main Models) ====================
    # Realistic models
    {'name': 'RealisticVision V6', 
     'url': 'https://civitai.com/api/download/models/501240', 
     'filename': 'RealisticVisionV6.safetensors', 
     'folder': 'checkpoints'},
]

# ============================================================================
# 🚀 DOWNLOAD PROCESS STARTS HERE (No need to edit below)
# ============================================================================

# Create directories
for item in downloads:
    folder_path = f'{base_dir}/models/{item["folder"]}'
    os.makedirs(folder_path, exist_ok=True)

print(f"📊 {len(downloads)} items to download\n")
successful = 0
failed = 0
skipped = 0

for i, item in enumerate(downloads, 1):
    folder_path = f'{base_dir}/models/{item["folder"]}'
    output_file = os.path.join(folder_path, item['filename'])
    
    print(f"\n[{i}/{len(downloads)}] {item['name']}")
    
    # Skip if already exists
    if os.path.exists(output_file):
        file_size = os.path.getsize(output_file) / (1024**2)  # MB
        if file_size > 1:  # File larger than 1MB, assume valid
            skipped += 1
            print(f"  ⏭️  Already exists ({file_size:.1f} MB)")
            continue
    
    # Download with progress bar
    if download_file_with_progress(item['url'], output_file):
        file_size = os.path.getsize(output_file) / (1024**2)
        successful += 1
        print(f"  ✅ Success ({file_size:.1f} MB)")
    else:
        failed += 1
        print(f"  ❌ Failed")
    
    time.sleep(0.5)

print("\n" + "="*60)
print(f"📊 Download Complete!")
print(f"   ✅ Successful: {successful}")
print(f"   ⏭️  Skipped (already exists): {skipped}")
print(f"   ❌ Failed: {failed}")
print("="*60)

In [ ]:
import os
import subprocess
import shutil

print("=" * 70)
print("Robust Download: Hunyuan3D-2 MV Models (Download to /tmp, then Move)")
print("=" * 70)

# --- Configuration ---
# This dictionary defines all paths for the two-step download and move process.
models_to_process = {
    "normal_multiview": {
        "url": "https://huggingface.co/tencent/Hunyuan3D-2mv/resolve/main/hunyuan3d-dit-v2-mv/model.fp16.safetensors",
        "tmp_path": "/tmp/hunyuan3d-dit-v2-mv.safetensors",
        "final_dir": "/kaggle/working/ComfyUI/models/diffusion_models",
        "final_filename": "hunyuan3d-dit-v2-mv.fp16.safetensors"
    }
}

# --- Main Logic ---
for name, model in models_to_process.items():
    print(f"\n--- Processing: {name} ---")
    
    final_path = os.path.join(model["final_dir"], model["final_filename"])
    os.makedirs(model["final_dir"], exist_ok=True)

    # STEP 0: Check if the file is already in its final destination
    if os.path.exists(final_path) and os.path.getsize(final_path) > 4 * 1024**3:
        file_size_gb = os.path.getsize(final_path) / (1024**3)
        print(f"✓ Model already exists in final destination ({file_size_gb:.2f} GB). Skipping.")
        continue

    # STEP 1: Download to /tmp directory
    print(f"Downloading to temporary path: {model['tmp_path']}")
    try:
        subprocess.run([
            'wget', '--no-check-certificate', '--continue', '--progress=dot:giga',
            '--timeout=900', '--tries=5', '--retry-connrefused',
            '-O', model['tmp_path'], model['url']
        ], check=True, timeout=1800)

        # Verify download integrity
        if not os.path.exists(model['tmp_path']) or os.path.getsize(model['tmp_path']) < 4 * 1024**3:
            raise Exception("Download completed, but file is incomplete.")
        
        print("✓ Temporary download successful.")

        # STEP 2: Move the completed file from /tmp to the final destination
        print(f"Moving file to: {final_path}")
        shutil.move(model['tmp_path'], final_path)
        print("✓ Move complete!")

    except Exception as e:
        print(f"✗ ERROR during {name} processing: {e}")

# --- Final Summary ---
print("\n" + "="*70)
print("DOWNLOAD & MOVE SUMMARY")
print("="*70)
for name, model in models_to_process.items():
    final_path = os.path.join(model["final_dir"], model["final_filename"])
    if os.path.exists(final_path):
        print(f"  ✓ {name}: Successfully placed at {final_path}")
    else:
        print(f"  ✗ {name}: FAILED to place model in final directory.")


# **Start ComfyUI**

In [ ]:
# Start ComfyUI with Zrok tunneling
import os
import subprocess

print("🚀 STARTING COMFYUI WITH ZROK TUNNELING")
print("=" * 50)

# Set environment variables to redirect large downloads to /tmp
os.environ['HF_HOME'] = '/tmp/huggingface'
os.environ['HF_HUB_CACHE'] = '/tmp/huggingface/hub'
os.environ['TRANSFORMERS_CACHE'] = '/tmp/huggingface/transformers'
os.environ['DIFFUSERS_CACHE'] = '/tmp/huggingface/diffusers'
os.environ['TORCH_HOME'] = '/tmp/torch'
os.environ['XDG_CACHE_HOME'] = '/tmp/cache'

print("📦 Cache directories redirected to /tmp:")
print(f"   HF_HOME: {os.environ['HF_HOME']}")
print(f"   DIFFUSERS_CACHE: {os.environ['DIFFUSERS_CACHE']}")
print(f"   TORCH_HOME: {os.environ['TORCH_HOME']}")
print(f"   XDG_CACHE_HOME: {os.environ['XDG_CACHE_HOME']}")
print("=" * 50)

# Determine correct Python environment
venv_python = '/kaggle/working/venv/bin/python'
system_python = '/usr/bin/python3'

if os.path.exists(venv_python):
    try:
        result = subprocess.run([venv_python, '--version'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            PYTHON_EXEC = venv_python
            print(f"✅ Using Virtual Environment: {venv_python}")
        else:
            PYTHON_EXEC = system_python
            print(f"⚠️  Using System Python: {system_python}")
    except:
        PYTHON_EXEC = system_python
        print(f"⚠️  Using System Python: {system_python}")
else:
    PYTHON_EXEC = system_python
    print(f"📝 Using System Python: {system_python}")

%cd /kaggle/working/ComfyUI
port = '8188'

print(f"🐍 Python: {PYTHON_EXEC}")
print(f"🌐 Port: {port}")
print("🔗 Zrok will provide external URL")
print("=" * 50)

# Start ComfyUI with Zrok tunneling
!chmod a+x /kaggle/working/zrok/zrok

cmd = f'{PYTHON_EXEC} /kaggle/working/ComfyUI/main.py --listen 0.0.0.0 --port {port} & /kaggle/working/zrok/zrok share public http://localhost:{port} --headless'
get_ipython().system(cmd)